In [ ]:
# For colab
import os
from google.colab import userdata

# 1. Pull the secrets using the interactive Colab kernel
colab_hf_token = userdata.get('HF_TOKEN')
colab_wandb_key = userdata.get('WANDB_API_KEY')

# 2. Inject them into the system environment variables
os.environ["HF_TOKEN"] = colab_hf_token
os.environ["WANDB_API_KEY"] = colab_wandb_key

print("✅ Secrets successfully injected into environment.")

In [ ]:
import subprocess, os, signal, time

SHUTDOWN_FILE = "/tmp/SHUTDOWN_REQUESTED"
USER_STOP_MARKER = "/tmp/USER_STOPPED_TRAINING"

# Clean stale files
for f in [SHUTDOWN_FILE, USER_STOP_MARKER, "/tmp/libtpu_lockfile"]:
    try: os.remove(f)
    except: pass
for i in range(8):
    try: os.remove(f"/tmp/heartbeat_rank_{i}.txt")
    except: pass

# Isolate training from notebook's SIGINT
proc = subprocess.Popen(
    ["python", "-u", "parallel_hardware_trainer.py"],
    start_new_session=True
)
print(f"Training started (PID {proc.pid}).\n")

try:
    proc.wait()
except KeyboardInterrupt:
    print("\n🛑 Stop pressed. Writing shutdown file...")
    with open(SHUTDOWN_FILE, "w") as f:
        f.write("user")
    with open(USER_STOP_MARKER, "w") as f:
        f.write("graceful")
    print("⏳ Waiting for workers to exit cleanly (up to 5 min)...")
    try:
        proc.wait(timeout=300)
        print(f"✅ Exited cleanly (code {proc.returncode}).")
    except KeyboardInterrupt:
        print("⚠️ Second stop — force killing.")
        try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except: pass
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        print("⚠️ Timed out — force killing.")
        try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except: pass
        proc.wait(timeout=10)

print(f"🏁 Done (code {proc.returncode}).")

In [ ]:
import subprocess, os, signal, time, sys

SHUTDOWN_FILE = "/tmp/SHUTDOWN_REQUESTED"
USER_STOP_MARKER = "/tmp/USER_STOPPED_TRAINING"
MAX_RETRIES = 5
BASE_BACKOFF = 30
MIN_RUNTIME = 60

# Clean stale files
for f in [SHUTDOWN_FILE, USER_STOP_MARKER, "/tmp/libtpu_lockfile"]:
    try: os.remove(f)
    except: pass
for i in range(8):
    try: os.remove(f"/tmp/heartbeat_rank_{i}.txt")
    except: pass

attempt = 0
while attempt < MAX_RETRIES:
    attempt += 1
    print(f"\n{'='*40}")
    print(f"🚀 Launch attempt {attempt}/{MAX_RETRIES}")
    print(f"{'='*40}\n")

    # Isolate training in its own session
    proc = subprocess.Popen(
        ["python", "-u", "parallel_hardware_trainer.py"],
        start_new_session=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0  # unbuffered binary
    )

    start = time.time()
    user_stopped = False

    try:
        # Read output line-by-line, decode to string, and flush to the notebook
        for line in iter(proc.stdout.readline, b''):
            sys.stdout.write(line.decode('utf-8', errors='replace'))
            sys.stdout.flush()
        proc.wait()

    except KeyboardInterrupt:
        # Stop button pressed — write shutdown file, wait for clean exit
        user_stopped = True
        print("\n🛑 Stop pressed. Writing shutdown file...")
        with open(SHUTDOWN_FILE, "w") as f:
            f.write("user")
        with open(USER_STOP_MARKER, "w") as f:
            f.write("graceful")

        # Drain remaining output while waiting for exit
        print("⏳ Waiting for clean exit (up to 5 min)...")
        try:
            remaining = proc.stdout.read()
            if remaining:
                sys.stdout.write(remaining.decode('utf-8', errors='replace'))
                sys.stdout.flush()
            proc.wait(timeout=300)
            print(f"✅ Exited cleanly (code {proc.returncode}).")
        except KeyboardInterrupt:
            print("⚠️ Second stop — force killing.")
            try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except ProcessLookupError: pass
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            print("⚠️ Timed out — force killing.")
            try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except ProcessLookupError: pass
            proc.wait(timeout=10)

    runtime = time.time() - start

    # User pressed stop → don't restart
    if user_stopped or os.path.exists(USER_STOP_MARKER):
        print("🛑 User-initiated stop. Not restarting.")
        try: os.remove(USER_STOP_MARKER)
        except: pass
        try: os.remove(SHUTDOWN_FILE)
        except: pass
        break

    # Clean exit → training finished
    if proc.returncode == 0:
        print(f"✅ Training finished after {runtime:.0f}s.")
        break

    # Crash → retry with backoff
    print(f"💥 Crashed (code {proc.returncode}) after {runtime:.0f}s.")

    if attempt >= MAX_RETRIES:
        print(f"⚠️ Hit max retries ({MAX_RETRIES}). Giving up.")
        break

    # Fast crash = exponential backoff, long run = short backoff
    if runtime < MIN_RUNTIME:
        backoff = BASE_BACKOFF * (2 ** (attempt - 1))
        print(f"⚠️ Fast crash — backing off {backoff}s...")
    else:
        backoff = BASE_BACKOFF
        print(f"⏳ Restarting in {backoff}s (resume from last checkpoint)...")

    # Clean stale files before retry
    for f in [SHUTDOWN_FILE, USER_STOP_MARKER, "/tmp/libtpu_lockfile"]:
        try: os.remove(f)
        except: pass

    time.sleep(backoff)

print("\n🏁 Launcher done.")